In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from processL0b.reader import Reader


r = Reader("C:\\Users\\agwhi\\Desktop\\cristal\\data413\\l0b\\26_04_13__17_42_51__LN2config1.h5")

c:\Users\agwhi\Desktop\Code\DataSystemPy3\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


For each radiometer row, we want to associate a temperature row and a gps row.
Find the closest system timestamp and tie those together.

In [ ]:
# This gets the radiometer rows between the ith and ith + 1 thermistor timestamp
# Each step contains about 5500 rows, but some rows contain double or even triple? Why is that?
# for i in range(len(r.thermistors.data) - 1):
i = 3
r.radiometer.data[
    (r.radiometer.data['Timestamp'] - r.thermistors.temps['Timestamp'][i] > 0)
    &
    (r.radiometer.data['Timestamp'] - r.thermistors.temps['Timestamp'][i+1] < 0)
]

In [2]:
# Get rows at nadir for each revolution
MOTOR_NADIR = [3500, 4000]

nadir_per_revolution = r.radiometer.data[
    (r.radiometer.data['MotorPosition'] > MOTOR_NADIR[0])
    &
    (r.radiometer.data['MotorPosition'] < MOTOR_NADIR[1])
].groupby('Revolution').mean()
nadir_per_revolution

,Timestamp,MotorPosition,34 QV,Not Connected 1,18 QV,24 QV,34 QH,Not Connected 2,18 QH,24 QH
Revolution,,,,,,,,,,
0,1.776124e+09,3813.254098,9834.024590,16237.155738,7604.672131,9729.926230,10944.377049,16311.704918,6880.844262,8499.442623
1,1.776124e+09,3750.297468,9837.139241,16237.734177,7600.500000,9735.310127,10944.063291,16311.822785,6868.784810,8498.037975
2,1.776124e+09,3750.624204,9834.993631,16237.484076,7598.082803,9738.923567,10943.707006,16311.643312,6871.993631,8498.050955
3,1.776124e+09,3749.528662,9834.681529,16237.732484,7592.624204,9737.063694,10943.828025,16311.496815,6862.974522,8500.878981
4,1.776124e+09,3749.483871,9835.032258,16237.516129,7587.774194,9736.006452,10943.193548,16311.683871,6860.606452,8493.606452
...,...,...,...,...,...,...,...,...,...,...
116,1.776124e+09,3750.115385,9841.070513,16237.916667,7621.333333,9753.038462,10951.705128,16311.987179,6880.724359,8517.147436
117,1.776124e+09,3749.496774,9844.896774,16237.245161,7617.341935,9750.696774,10950.412903,16311.632258,6887.361290,8523.600000
118,1.776124e+09,3748.910256,9845.506410,16237.480769,7622.987179,9755.570513,10953.141026,16311.897436,6888.525641,8521.384615


In [4]:
# Average a row per revolution between the given values
MOTOR_ZENITH = [11500, 12000]

zenith_per_revolution = r.radiometer.data[
    (r.radiometer.data['MotorPosition'] > MOTOR_ZENITH[0])
    &
    (r.radiometer.data['MotorPosition'] < MOTOR_ZENITH[1])
].groupby('Revolution').mean()

CALTAR_THERMISTORS = [9, 10, 11, 12, 13, 14, 15, 16]

def find_closest_index(series, value):
    return series.iloc[
        (series - value).abs().argsort()[:1]
    ].index[0]

i = []
for timestamp in zenith_per_revolution['Timestamp']:
    i.append( find_closest_index(r.thermistors.data['Timestamp'], timestamp) )
means = r.thermistors.data.iloc[i][CALTAR_THERMISTORS].mean(axis=1).reset_index(drop=True)

zenith_per_revolution['Temps'] = means
zenith_per_revolution

,Timestamp,MotorPosition,34 QV,Not Connected 1,18 QV,24 QV,34 QH,Not Connected 2,18 QH,24 QH,Temps
Revolution,,,,,,,,,,,
0,1.776124e+09,11749.400000,9313.961290,16237.561290,6507.548387,8870.767742,10599.000000,16311.516129,5490.929032,7345.483871,262.276112
1,1.776124e+09,11750.162338,9312.746753,16237.487013,6502.831169,8870.577922,10602.402597,16311.636364,5494.474026,7340.331169,262.276112
2,1.776124e+09,11749.935484,9314.154839,16237.535484,6513.761290,8874.819355,10598.832258,16311.832258,5491.825806,7348.858065,262.277153
3,1.776124e+09,11751.455128,9315.788462,16237.608974,6517.794872,8873.692308,10599.858974,16311.935897,5494.019231,7346.282051,262.032152
4,1.776124e+09,11751.261146,9313.254777,16237.414013,6512.993631,8870.280255,10601.630573,16311.808917,5490.554140,7345.910828,262.032152
...,...,...,...,...,...,...,...,...,...,...,...
115,1.776124e+09,11749.211538,9325.525641,16237.884615,6568.384615,8897.455128,10610.980769,16311.570513,5537.275641,7374.852564,262.051343
116,1.776124e+09,11750.301282,9330.634615,16237.942308,6557.044872,8899.326923,10612.980769,16312.076923,5532.794872,7380.724359,262.000601
117,1.776124e+09,11750.356688,9327.363057,16237.152866,6556.356688,8897.407643,10613.611465,16311.369427,5539.401274,7380.898089,262.000601


In [11]:
# For each channel, gain and offset are determined
LN2_TEMP = 77
labels = [ch.label for ch in r.radiometer.channels]
df = pd.DataFrame({
    'zenith': zenith_per_revolution[labels].mean(),
    'nadir': nadir_per_revolution[labels].mean(),
})
df['gain'] = (df['zenith'] - df['nadir']) / (zenith_per_revolution['Temps'].mean() - LN2_TEMP)
df['offset'] = df['zenith'] - df['gain'] * zenith_per_revolution['Temps'].mean()
# df = df.transpose()
r.radiometer.data['34 QV'] * df.loc['34 QV', 'gain']
# tb = (r.radiometer.data[labels] - df.loc['offset']) / df.loc['gain']

# for ch in r.radiometer.channels:
#     if ch.frequency == 0:
#         continue
#     plt.plot(
#         r.radiometer.data['MotorPosition'], tb[ch.label], linestyle='none', marker='.',
#         label=ch.label,
#     )
# plt.legend()
# plt.show()


0        -27380.246271
1        -27556.100641
2        -27489.108500
3        -27480.734482
4        -27427.699037
              ...     
600056   -27008.998156
600057   -27053.659584
600058   -27006.206817
600059   -27036.911549
600060   -26986.667443
Name: 34 QV, Length: 600061, dtype: float64